<style>
body {
    font-size: 20pt !important;
}

.rendered_html {
    font-size: 20pt !important;
}

.CodeMirror pre {
    font-size: 20pt !important;
}

.output pre {
    font-size: 20pt !important;
}
</style>


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data

In [2]:
import numpy as np
from astropy.table import Table
from astropy.io import ascii
from scipy.spatial import Delaunay
import pandas as pd
from itertools import combinations
import networkx as nx

In [3]:
filt_n1 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt1.ecsv").to_pandas()
filt_n2 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt2.ecsv").to_pandas()

In [4]:
n_random = 10
class_ngc1 = {}
for i in range(n_random):
    table = Table.read(f"/content/drive/MyDrive/DESI/classification/LRG_NGC_1_random_{i}_filt.fits").to_pandas()
    class_ngc1[i] = dict(zip(table['TARGETID'], table[f'class_{i}']))

class_ngc2 = {}
for i in range(n_random):
    table = Table.read(f"/content/drive/MyDrive/DESI/classification/LRG_NGC_2_random_{i}_filt.fits").to_pandas()
    class_ngc2[i] = dict(zip(table['TARGETID'], table[f'class_{i}']))

In [5]:
def neighbors(data_all, output_path=None):
    df_tri = data_all[['x', 'y', 'z']].values
    tri = Delaunay(df_tri)

    G = nx.Graph()
    ids = data_all['TARGETID'].values
    types = data_all['type'].values

    for coords, tipo, id_ in zip(df_tri, types, ids):
        G.add_node(id_, type=tipo)

    G.add_edges_from(
        (ids[s[i]], ids[s[j]])
        for s in tri.simplices
        for i in range(3)
        for j in range(i + 1, 4)
    )

    degree_dict = dict(G.degree())
    data_all['degree'] = data_all['TARGETID'].map(degree_dict)

    neighbor_ids_data = []
    neighbor_ids_rand = []

    for node in ids:
        neighbors = list(G.neighbors(node))
        d_ids = [n for n in neighbors if G.nodes[n]['type'] == 'data']
        r_ids = [n for n in neighbors if G.nodes[n]['type'] == 'rand']

        neighbor_ids_data.append(d_ids)
        neighbor_ids_rand.append(r_ids)

    result = pd.DataFrame({
        'TARGETID': ids,
        'degree': data_all['degree'].values,
        'type': data_all['type'].values,
        'neighbor_ids_data': neighbor_ids_data,
        'neighbor_ids_rand': neighbor_ids_rand
    })

    return result

In [6]:
def classification(data,number_rand):

    data.loc[(data['r'] >= -1.0) & (data['r'] <= -0.9), f'class_{number_rand}'] = 'void'
    data.loc[(data['r'] >  -0.9) & (data['r'] <=  0.0), f'class_{number_rand}'] = 'sheet'
    data.loc[(data['r'] >   0.0) & (data['r'] <=  0.9), f'class_{number_rand}'] = 'filament'
    data.loc[(data['r'] >   0.9) & (data['r'] <=  1.0), f'class_{number_rand}'] = 'knot'

    data.sort_values('z', inplace=True)

    return data

In [7]:
%%time
#data = {'1':filt_n1,'2':filt_n2}
data = {'1':filt_n1}

n_random = 1

for key, tbl in data.items():

    tbl = tbl.copy()
    tbl['type'] = 'data'

    for j in range(n_random):
        print(f'random {j}')
        df_rand = Table.read(f'/content/drive/MyDrive/DESI/rand/LRG_NGC_{key}_random_{j}_filt.fits').to_pandas()
        df_rand['type'] = 'rand'

        df_concat = pd.concat([df_rand, tbl], ignore_index=True)

        data_with_r = neighbors(df_concat)

        ids_validos_data = set(filt_n1['TARGETID'])
        ids_validos_rand = set(df_rand['TARGETID'])

        ids_validos_totales = ids_validos_data.union(ids_validos_rand)

        # Verificar los resultados de neighbors()
        for idx, row in data_with_r.iterrows():
            if row['TARGETID'] not in ids_validos_totales:
                print(f"TARGETID no encontrado: {row['TARGETID']}")
            for n in row['neighbor_ids_data']:
                if n not in ids_validos_totales:
                    print(f" Vecino data no encontrado: {n}")
            for n in row['neighbor_ids_rand']:
                if n not in ids_validos_totales:
                    print(f"Vecino rand no encontrado: {n}")


        #data_with_r[i][f'class_{i}'] = data_with_r[i]['TARGETID'].map(class_ngc1[i])

        #final_data = data_with_r[['TARGETID', 'N_random', 'N_data']]
        #final_data = data_with_r[['TARGETID', 'neighbor_ids_data', 'neighbor_ids_rand']]
        print(data_with_r.head())

        filename = f"/content/drive/MyDrive/DESI/connections/LRG_NGC_{key}_random_{j}_neighbors.parquet"
        data_with_r.to_parquet(filename, index=False)

random 0
             TARGETID  degree  type  \
0  327858307949660901      20  rand   
1  327858067003673263      17  rand   
2  327858279755548320      11  rand   
3  327858055830046125      15  rand   
4  327858134552938296      17  rand   

                                   neighbor_ids_data  \
0  [39627931789560151, 39627931797950869, 3962793...   
1  [39627696883370754, 39627702918974195, 3962769...   
2             [39627891553604133, 39627903603839062]   
3             [39627685701355800, 39627673646929302]   
4  [39627758401227765, 39627764441026960, 3962776...   

                                   neighbor_ids_rand  
0  [327858313955901760, 327858301930832116, 32785...  
1  [327858079066491876, 327858066999476247, 32785...  
2  [327858273736720458, 327858279763936010, 32785...  
3  [327858067892863786, 327858061865650619, 32785...  
4  [327858128513141629, 327858128513139254, 32785...  
CPU times: user 7min 1s, sys: 7.57 s, total: 7min 9s
Wall time: 7min 24s
